# 轻量级 PRM 完整实验流程

这个 notebook 从数据偏差审计开始，依次完成五种 reward head 训练、标准指标、分层与错误边界分析、确定性扰动、因果前缀检查、离线启发式剪枝和最终汇总。默认不会重新运行 LLM encoder。

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

PROJECT_ROOT = Path.cwd()

# 如果你的 precompute cache 目录不是这两个路径，改这里即可。
TRAIN_CACHE = PROJECT_ROOT / "cache" / "train"
VAL_CACHE = PROJECT_ROOT / "cache" / "val"

CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
RESULTS_DIR = PROJECT_ROOT / "results"

EPOCHS = 10
LR = 1e-3
ZETA = 4.0
SEED = 42
EARLY_STOPPING_PATIENCE = 2
CALIBRATION_FRACTION = 0.5
RUN_PERTURBATIONS = True
PRUNING_BUDGETS = [0.01, 0.05, 0.10]
PRIMARY_PRUNING_BUDGET = 0.05
PRIMARY_PRUNING_POLICY = "single_low"
BOOTSTRAP_SAMPLES = 1000

# 完整 sweep 会训练全部 head；如果只想比较旧 reward head 和 attention，可改成 ["linear", "attention"]。
HEADS = ["linear", "mlp", "cnn", "gru", "attention"]
COMPARE_HEADS = HEADS + ["majority", "position_only", "coin_flip"]
BASELINE_TRIALS = 100

CHECKPOINT_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

def run(cmd):
    print("\n$ " + " ".join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), check=True)

def inspect_cache(cache_dir):
    cache_dir = Path(cache_dir)
    hidden_file = cache_dir / "hidden_size.txt"
    shards = sorted(cache_dir.glob("shard_*.pt"))
    if not hidden_file.exists():
        raise FileNotFoundError(f"Missing {hidden_file}")
    if not shards:
        raise FileNotFoundError(f"No shard_*.pt files in {cache_dir}")
    print(f"{cache_dir}: hidden_size={hidden_file.read_text().strip()}, shards={len(shards)}")

print("Project:", PROJECT_ROOT)
print("Train cache:", TRAIN_CACHE)
print("Val cache:", VAL_CACHE)
inspect_cache(TRAIN_CACHE)
inspect_cache(VAL_CACHE)

## 1. 数据与位置偏差审计

在训练结果比较之前，先计算标签分布、相对位置错误率，以及 majority / position-only 确定性 baseline。

In [ ]:
run([
    sys.executable, "-m", "analysis.analyze_data_bias",
    "--cache_dir", VAL_CACHE,
    "--results_dir", RESULTS_DIR,
    "--calibration_fraction", CALIBRATION_FRACTION,
    "--split_seed", SEED,
])

## 2. 训练 reward heads 并保存 loss 曲线

每个 head 会输出：

- `checkpoints/{head}_head.pt`
- `results/{head}_efficiency.json`
- `results/{head}_loss_history.json`
- `results/{head}_loss_curve.png`，包含 train loss 和 evaluation loss

In [ ]:
for head in HEADS:
    run([
        sys.executable, "train_from_cache.py",
        "--cache_dir", TRAIN_CACHE,
        "--val_cache_dir", VAL_CACHE,
        "--head", head,
        "--epochs", EPOCHS,
        "--lr", LR,
        "--zeta", ZETA,
        "--seed", SEED,
        "--early_stopping_patience", EARLY_STOPPING_PATIENCE,
        "--calibration_fraction", CALIBRATION_FRACTION,
        "--split_seed", SEED,
        "--save_path", CHECKPOINT_DIR / f"{head}_head.pt",
        "--results_dir", RESULTS_DIR,
    ])

## 3. 查看 loss 曲线

In [ ]:
from IPython.display import Image, Markdown, display

for head in HEADS:
    path = RESULTS_DIR / f"{head}_loss_curve.png"
    if path.exists():
        print(head)
        display(Image(filename=str(path)))

## 4. Step-level evaluation

使用验证集 cache 计算 step reward accuracy 和 Q-value ranking accuracy。

In [ ]:
for head in HEADS:
    run([
        sys.executable, "-m", "eval.eval_step_metrics",
        "--cache_dir", VAL_CACHE,
        "--head", head,
        "--checkpoint", CHECKPOINT_DIR / f"{head}_head.pt",
        "--calibration_fraction", CALIBRATION_FRACTION,
        "--split_seed", SEED,
        "--results_dir", RESULTS_DIR,
    ])

## 5. 分层、首错边界与扰动分析

先复用验证 cache 生成逐步 Q-value，再比较轨迹长度、首错位置、错误数量及位置匹配后的首错边界效应。可选扰动只运行轻量 head，不重新编码或训练。

In [ ]:
behavior_cmd = [
    sys.executable, "-m", "analysis.analyze_head_behavior",
    "--cache_dir", VAL_CACHE,
    "--checkpoint_dir", CHECKPOINT_DIR,
    "--results_dir", RESULTS_DIR,
    "--heads", *HEADS,
    "--calibration_fraction", CALIBRATION_FRACTION,
    "--split_seed", SEED,
]
if RUN_PERTURBATIONS:
    behavior_cmd.append("--run_perturbations")
run(behavior_cmd)

for filename in ["behavior_by_group.png", "first_error_boundary.png", "perturbation_sensitivity.png"]:
    path = RESULTS_DIR / filename
    if path.exists():
        display(Image(filename=str(path)))

## 6. 因果前缀检查与离线启发式剪枝

对第 `t` 步只输入前 `t` 个缓存 step embedding，避免上下文 head 使用未来步骤。阈值只在 calibration 半区的全正确轨迹上确定，然后在 held-out test 上报告误剪率、错误检出、检测延迟和安全步骤节省率。该过程不运行 Qwen，也不生成新文本。

In [ ]:
run([
    sys.executable, "-m", "analysis.analyze_offline_pruning",
    "--cache_dir", VAL_CACHE,
    "--checkpoint_dir", CHECKPOINT_DIR,
    "--results_dir", RESULTS_DIR,
    "--heads", *HEADS,
    "--budgets", *PRUNING_BUDGETS,
    "--primary_budget", PRIMARY_PRUNING_BUDGET,
    "--primary_policy", PRIMARY_PRUNING_POLICY,
    "--bootstrap_samples", BOOTSTRAP_SAMPLES,
    "--calibration_fraction", CALIBRATION_FRACTION,
    "--split_seed", SEED,
    "--seed", SEED,
])

for filename in ["full_vs_causal_scores.png", "pruning_tradeoff.png", "pruning_detection_delay.png"]:
    path = RESULTS_DIR / filename
    if path.exists():
        display(Image(filename=str(path)))

pruning_summary = RESULTS_DIR / "pruning_summary.md"
if pruning_summary.exists():
    display(Markdown(pruning_summary.read_text()))

## 7. 可选：single-eval cache 评估

如果已经有 `cache/single_eval`，这里会继续跑最终解答级别的准确率和 separation；没有就自动跳过。

In [ ]:
SINGLE_EVAL_CACHE = PROJECT_ROOT / "cache" / "single_eval"

if (SINGLE_EVAL_CACHE / "hidden_size.txt").exists() and list(SINGLE_EVAL_CACHE.glob("shard_*.pt")):
    for head in HEADS:
        run([
            sys.executable, "-m", "eval.eval_single_from_cache",
            "--cache_dir", SINGLE_EVAL_CACHE,
            "--head", head,
            "--checkpoint", CHECKPOINT_DIR / f"{head}_head.pt",
            "--calibration_fraction", CALIBRATION_FRACTION,
            "--split_seed", SEED,
            "--results_dir", RESULTS_DIR,
        ])
else:
    print(f"Skip single-eval: no cache found at {SINGLE_EVAL_CACHE}")

## 8. 随机掷硬币 baseline

这个 baseline 不训练模型，随机猜每一步或每条推理是否正确，并写入 `results/coin_flip_*.json`，用于最终表格比较。

In [ ]:
baseline_cmd = [
    sys.executable, "-m", "eval.eval_coin_flip_baseline",
    "--val_cache_dir", VAL_CACHE,
    "--results_dir", RESULTS_DIR,
    "--trials", BASELINE_TRIALS,
    "--seed", SEED,
    "--calibration_fraction", CALIBRATION_FRACTION,
    "--split_seed", SEED,
]
if (SINGLE_EVAL_CACHE / "hidden_size.txt").exists() and list(SINGLE_EVAL_CACHE.glob("shard_*.pt")):
    baseline_cmd.extend(["--single_cache_dir", SINGLE_EVAL_CACHE])
run(baseline_cmd)

## 9. 汇总结果表

In [ ]:
run([
    sys.executable, "-m", "eval.summarize_results",
    "--results_dir", RESULTS_DIR,
    "--out_csv", RESULTS_DIR / "summary.csv",
    "--out_md", RESULTS_DIR / "summary.md",
])

summary_md = RESULTS_DIR / "summary.md"
if summary_md.exists():
    print(summary_md.read_text())

## 10. 架构、确定性 baseline 与随机 baseline 对比

In [ ]:
import csv
from IPython.display import Markdown, display

summary_csv = RESULTS_DIR / "summary.csv"
if summary_csv.exists():
    rows = list(csv.DictReader(summary_csv.open()))
    available = set(COMPARE_HEADS)
    cols = [
        "head",
        "final_train_loss",
        "final_eval_loss",
        "step_reward_accuracy",
        "step_balanced_accuracy",
        "step_roc_auc",
        "step_average_precision",
        "qvalue_ranking_accuracy",
        "position_controlled_boundary_effect",
        "causal_step_roc_auc",
        "clean_false_prune_rate",
        "error_coverage",
        "detection_at_0",
        "median_detection_delay",
        "safe_step_saving_rate",
        "oracle_efficiency_ratio",
        "single_eval_accuracy",
        "single_eval_roc_auc",
        "single_eval_separation",
        "n_trainable_params",
        "total_train_time_sec",
    ]
    cols = [c for c in cols if rows and c in rows[0]]
    selected = [row for row in rows if row["head"] in available]
    if selected:
        table = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
        table.extend("| " + " | ".join(row.get(c, "") for c in cols) + " |" for row in selected)
        display(Markdown("\n".join(table)))
    else:
        print("No comparison heads found in summary.csv")
else:
    print("Run the summary cell first.")